# Two Representations, Three Formats

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

A tree read by the package is held twice over, in a sense: as a `TreeData`,
which keeps the branches as NumPy arrays exactly as GATE wrote them, and as a
pandas frame, which is what an analysis selects, groups and plots. This
notebook looks at both side by side — what each one holds, what passes from one
to the other, and what does not come back.

It then writes the same data in the three formats the package offers — ROOT,
HDF5 and CSV — and reads each file back, to see which of them keeps the shape
of the data and which keeps only the numbers. The question it answers is which
representation and which format to reach for, and what each one costs.

## What will you learn from this notebook

- how to read what a `TreeData` holds: entries, branches, types, array branches
- how to take a subset of branches without copying any data
- why two `TreeData` are compared branch by branch rather than with `==`
- what happens to an array branch such as `volumeID` in the pandas view, and
  why it does not come back on its own
- how to name an output file the way the command line does
- which of ROOT, HDF5 and CSV keeps the array branches and the data types, and
  how to read each file back

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The file read here comes from a scanner described with a `cylindricalPET`
system, so besides the usual branches its hits carry the identifiers of the
scanner's volumes. What it holds is described in
[`../data/README.md`](../data/README.md).

In [2]:
output = paths.output_dir("tutorial-03-representations-and-formats")
example_input = paths.input_example("hits-a2-cylindrical-pet.root")

In [3]:
import h5py
import numpy as np
import pandas as pd

from opengate_gate_tree import (
    BranchNotFoundError,
    GateTree,
    GateTreeError,
    OutputFileFormat,
    TreeData,
    build_output_file_path,
    read_tree,
    write_tree,
)

### The NumPy View

Four branches: two numbers per hit, one text, and `volumeID`, which is not one
number per hit but ten.

In [4]:
hits = read_tree(example_input, GateTree.HITS, ["eventID", "edep", "processName", "volumeID"])
hits

TreeData(tree='Hits', entries=500, branches=4)

`len` and `entry_count` give the same answer; `branch_names` keeps the order
the branches were asked for:

In [5]:
len(hits), hits.entry_count, hits.branch_names

(500, 500, ('eventID', 'edep', 'processName', 'volumeID'))

The types are the ones GATE wrote, unchanged: 32-bit integers and floats,
and `object` for text, which NumPy holds as Python strings.

In [6]:
dict(hits.dtypes)

{'eventID': dtype('int32'),
 'edep': dtype('float32'),
 'processName': dtype('O'),
 'volumeID': dtype('int32')}

`array_branches` is the quickest way to tell the two kinds of column apart.
It names every branch that holds more than one number per entry, with its
width:

In [7]:
dict(hits.array_branches)

{'volumeID': 10}

In [8]:
hits["volumeID"].shape, hits["volumeID"][:3]

((500, 10),
 array([[ 0,  0, 91,  1,  0, 11,  0, -1, -1, -1],
        [ 0,  0, 72,  6,  4, 16,  0, -1, -1, -1],
        [ 0,  0, 65,  3,  0, 23,  0, -1, -1, -1]], dtype=int32))

One row per hit, ten identifiers per row. In this file the first seven
places carry identifiers of the volumes a hit happened in, and the last three
hold `-1` in every row: the geometry is not that deep.

### Taking A Subset Of Branches

`select` returns a new `TreeData` with the branches asked for, in the order
asked for. A name given twice is kept once, at its first position:

In [9]:
energies = hits.select(["edep", "eventID", "edep"])
energies.branch_names

('edep', 'eventID')

Nothing is copied. The new `TreeData` refers to the very arrays the first
one holds, so a subset of a large tree costs no memory:

In [10]:
np.shares_memory(energies["edep"], hits["edep"])

True

The other side of that coin: the arrays stay writable, so changing one in
place changes what every `TreeData` sharing it sees. Copy an array with
`.copy()` before modifying it.

What cannot change is the set of branches. The mapping of columns is
read-only, so a branch is neither added nor replaced after the fact:

In [11]:
try:
    hits.columns["edep"] = hits["edep"] * 1000
except TypeError as error:
    print(f"TypeError: {error}")

TypeError: 'mappingproxy' object does not support item assignment


A name the data does not hold is refused with the list of what it does
hold:

In [12]:
try:
    hits.select(["edep", "energy"])
except BranchNotFoundError as error:
    print(error)

Branches not found: ['energy']. Available branches: ['eventID', 'edep', 'processName', 'volumeID'].


### Why `==` Does Not Compare The Data

Comparing two NumPy arrays with `==` answers with an array, not with `True` or
`False`, so `TreeData` deliberately defines no comparison of its contents. What
`==` falls back to is Python's default: whether the two are the same object.
A subset holding every branch is a new object, so it is not "equal" to the
data it was taken from, although it holds the same arrays:

In [13]:
everything = hits.select(hits.branch_names)
hits == hits, everything == hits

(True, False)

Comparing data means comparing the branches that matter:

In [14]:
all(np.array_equal(everything[name], hits[name]) for name in hits.branch_names)

True

### The pandas View

`to_dataframe()` gives a plain `pandas.DataFrame`. A pandas cell holds one
value, so `volumeID` is spread over ten columns, `volumeID_0` to `volumeID_9`,
standing where the branch stood:

In [15]:
frame = hits.to_dataframe()
frame.head()

,eventID,edep,processName,volumeID_0,volumeID_1,volumeID_2,volumeID_3,volumeID_4,volumeID_5,volumeID_6,volumeID_7,volumeID_8,volumeID_9
0,0,0.511000,PhotoElectric,0,0,91,1,0,11,0,-1,-1,-1
1,0,0.511000,PhotoElectric,0,0,72,6,4,16,0,-1,-1,-1
2,2,0.511000,PhotoElectric,0,0,65,3,0,23,0,-1,-1,-1
3,2,0.327841,Compton,0,0,84,4,4,3,0,-1,-1,-1
4,2,0.183159,PhotoElectric,0,0,84,4,4,3,0,-1,-1,-1


In [16]:
frame.columns.tolist()

['eventID',
 'edep',
 'processName',
 'volumeID_0',
 'volumeID_1',
 'volumeID_2',
 'volumeID_3',
 'volumeID_4',
 'volumeID_5',
 'volumeID_6',
 'volumeID_7',
 'volumeID_8',
 'volumeID_9']

The frame is built from the arrays rather than over them: with the pandas
installed here, its columns do not share memory with the `TreeData`, so
working on the frame leaves the data that was read untouched.

In [17]:
pd.__version__, np.shares_memory(frame["edep"].to_numpy(), hits["edep"])

('3.0.5', False)

The way back is `TreeData.from_dataframe`. Every column becomes a scalar
branch: the ten columns are not folded back into one array branch, because
nothing tells ten columns that were once an array from ten that never were.

In [18]:
restored = TreeData.from_dataframe(GateTree.HITS, frame)
restored, dict(restored.array_branches)

(TreeData(tree='Hits', entries=500, branches=13), {})

The expansion is one way. When an array branch has to survive, keep it in
the `TreeData` and select rows there, as tutorial 02 shows in "Keeping An Array
Branch".

### Building Data Yourself

`TreeData` can also be built from arrays computed in the analysis — a derived
quantity to write next to the rest. Construction checks that every branch holds
the same number of entries:

In [19]:
try:
    TreeData(GateTree.HITS, {"eventID": hits["eventID"], "edep": hits["edep"][:10]})
except ValueError as error:
    print(error)

Branch 'edep' has 10 entries but 500 were expected; all branches must hold the same number of entries.


### Naming An Output File

Output files are not read back by GATE. They feed the next step of an
analysis, so the format to pick is the one that step reads most comfortably.
Whichever it is, `build_output_file_path` names the file the way the command
line does — `<title>.<tree>.<format>` — so that files written from a notebook
and from a script sit side by side:

In [20]:
outputs = {
    file_format: build_output_file_path(output, "scanner", GateTree.HITS, file_format)
    for file_format in OutputFileFormat
}
[path.name for path in outputs.values()]

['scanner.hits.root', 'scanner.hits.hdf5', 'scanner.hits.csv']

### Writing The Same Data Three Times

In [21]:
for file_format, path in outputs.items():
    write_tree(hits, path, file_format)
    print(f"{path.name:20} {path.stat().st_size:>7} bytes")

scanner.hits.root      29746 bytes
scanner.hits.hdf5      52480 bytes
scanner.hits.csv       24815 bytes


Each file is now read back in the way the next step of an analysis would
read it.

#### ROOT

The tree is a `TTree` named `Hits`, with `volumeID` an array branch of ten
integers and every type as it was. A selection of four branches is not a whole
"Hits" structure, so the structure check is turned off to read it back:

In [22]:
from_root = read_tree(outputs[OutputFileFormat.ROOT], GateTree.HITS, validate=False)
dict(from_root.array_branches), dict(from_root.dtypes)

({'volumeID': 10},
 {'eventID': dtype('int32'),
  'edep': dtype('float32'),
  'processName': dtype('O'),
  'volumeID': dtype('int32')})

#### HDF5

The tree becomes a group holding one dataset per branch. The group records
which tree it came from, how many entries it holds, the order of its branches —
which HDF5 would otherwise list alphabetically — and the version of the package
that wrote it:

In [23]:
with h5py.File(outputs[OutputFileFormat.HDF5]) as stored:
    group = stored["Hits"]
    print(list(group))
    print(dict(group.attrs))
    print(group["volumeID"].shape, group["edep"].dtype)
    print(group["processName"][:2])
    print(group["processName"].asstr()[:2])

['eventID', 'edep', 'processName', 'volumeID']
{'gate_tree': 'Hits', 'entries': np.int64(500), 'branches': array(['eventID', 'edep', 'processName', 'volumeID'], dtype=object), 'package_version': '0.5.0'}
(500, 10) float32
[b'PhotoElectric' b'PhotoElectric']
['PhotoElectric' 'PhotoElectric']


`volumeID` is kept as a `(500, 10)` dataset and `edep` as 32-bit floats.
Text is stored as UTF-8 strings: read directly it comes back as bytes, and
through `asstr()` as Python strings.

#### CSV

CSV has no cell for an array, so `volumeID` is written as ten columns, the
same ones the pandas view makes:

In [24]:
from_csv = pd.read_csv(outputs[OutputFileFormat.CSV])
from_csv.columns.tolist()

['eventID',
 'edep',
 'processName',
 'volumeID_0',
 'volumeID_1',
 'volumeID_2',
 'volumeID_3',
 'volumeID_4',
 'volumeID_5',
 'volumeID_6',
 'volumeID_7',
 'volumeID_8',
 'volumeID_9']

And text has no types, so pandas guesses them again when it reads the file:
64-bit integers and floats where GATE wrote 32-bit ones.

In [25]:
from_csv.dtypes[["eventID", "edep", "volumeID_0"]].to_dict()

{'eventID': dtype('int64'),
 'edep': dtype('float64'),
 'volumeID_0': dtype('int64')}

The numbers themselves survive. `0.511` written by GATE as a 32-bit float is
`0.5109999775886536` when widened to 64 bits; the CSV holds the shortest text
that reads back as the same 32-bit value, so the column read from the file
differs from the original as 64-bit floats, but equals it once it is narrowed
back to 32 bits:

In [26]:
csv_edep = from_csv["edep"].to_numpy()

(
    bool((csv_edep == hits["edep"]).all()),
    np.array_equal(csv_edep.astype(np.float32), hits["edep"]),
)

(False, True)

So the three formats, on this data:

| | ROOT | HDF5 | CSV |
| --- | --- | --- | --- |
| scalar branches | kept | kept | kept |
| text branch | kept | kept, read with `asstr()` | kept |
| `volumeID`, `(500, 10)` | kept as an array | kept as an array | ten columns |
| 32-bit types | kept | kept | re-inferred as 64-bit |

ROOT for an analysis in C++ with the ROOT framework, HDF5 for one in Python or
MATLAB, CSV for a look in a spreadsheet or a quick `read_csv`.

### What A Format Refuses

Some data has no faithful form in a format, and the writer refuses it rather
than writing something else. A tree without branches cannot be expressed in
ROOT; like every failure of reading or writing, the error derives from
`GateTreeError`:

In [27]:
try:
    write_tree(TreeData(GateTree.HITS, {}), output / "empty.hits.root", OutputFileFormat.ROOT)
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

ExportError: A ROOT tree cannot be written without branches.


A branch name holding a slash is refused by HDF5, because h5py would make a
nested group of it instead of a dataset:

In [28]:
slashed = TreeData(GateTree.HITS, {"cylindricalPET/gantryID": hits["eventID"]})

try:
    write_tree(slashed, output / "slashed.hits.hdf5", OutputFileFormat.HDF5)
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

ExportError: Branches cannot be written to a 'hdf5' file because their names hold one of "/": ['cylindricalPET/gantryID']. A slash makes h5py create a nested group instead of a dataset, which writes a file whose layout is not the one the data describes. GateToTree gives branches such names when a simulation uses more than one system.


## Summary

A `TreeData` keeps the branches as GATE wrote them, array branches included,
and shares its arrays rather than copying them; the pandas view spreads an
array branch over columns and does not fold it back. ROOT and HDF5 keep both
the arrays and the types, CSV keeps the numbers only — so the format follows
from what the next step of the analysis reads and from whether the shape of the
data has to survive.

## Bibliography

- [Data representations](https://opengate-gate-tree.readthedocs.io/en/latest/guide/representations.html)
  — `TreeData`, its pandas view, and why equality is not defined
- [Exporting files](https://opengate-gate-tree.readthedocs.io/en/latest/guide/export.html)
  — what each format preserves, the names a format cannot carry, and how a
  file is named
- [Using the package as a library](https://opengate-gate-tree.readthedocs.io/en/latest/guide/library.html)
  — the public API and the errors it raises
- [The "Hits" Tree](https://opengate-gate-tree.readthedocs.io/en/latest/guide/hits.html)
  — the `cylindricalPET` identifiers and the structure check

## Changelog

- v1: written.